# Experimento completo no Colab

Roda seis modelos em sequência: os principais (Qwen3.5-9B, Ministral 3 8B), o par de especialização em português (Lumi-4B × Nemotron 3 Nano 4B) e as comparações de tamanho (Ministral 3 14B e Qwen3.5-27B).

1. **Ambiente de execução > Alterar o tipo de ambiente > G4 GPU** (RTX PRO 6000, 96 GB). Na falta dela, A100 (roda todos) ou L4 (pula o Qwen3.5-27B). A T4 não serve: não roda FP8.
2. Segredo `HF_TOKEN` (ícone de chave na barra lateral) com acesso liberado ao notebook.
3. Licença aceita em https://huggingface.co/meta-llama/Llama-Prompt-Guard-2-86M.
4. Rode as células em ordem. Os resultados vão para o Google Drive, em `MyDrive/pfc-results`, ao fim de cada modelo.

**Se a sessão cair:** rode tudo de novo. Os modelos já completos no Drive são pulados.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!df -h /content | tail -1

In [ ]:
%cd /content
![ -d pfc ] || git clone https://github.com/luizfernandofo/pfc.git
%cd /content/pfc
!git pull

Instalação (3 a 5 minutos). Se o Colab pedir para reiniciar a sessão, reinicie e continue na próxima célula.

In [ ]:
%cd /content/pfc
!pip install -q -r requirements.txt

In [ ]:
import os
from google.colab import drive, userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
drive.mount('/content/drive')
%cd /content/pfc
!python -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__, '| GPU', torch.cuda.get_device_name(0))"

Execução completa. Os pesos são baixados modelo a modelo e apagados depois de cada um, para não lotar o disco. Ao fim de cada modelo sai um resumo com a % de acerto (perguntas limpas) e de sucesso do ataque, por vetor, idioma e defesa.

In [ ]:
%cd /content/pfc
!python src/run_all.py data/main --copiar /content/drive/MyDrive/pfc-results 2>&1 | grep -vE '^(INFO|WARNING|DEBUG)|it/s'

In [ ]:
!ls -la /content/drive/MyDrive/pfc-results